In [3]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import torch
from hydra.utils import instantiate
from hydra import initialize, compose
import hydra

import wandb

from data.dataManager import DataManager
from model.modelCreator import ModelCreator
from omegaconf import OmegaConf
from scripts.run import setup_model, load_model_instance

from utils.displays.shower_plots import visualize_low_ratio_events
from utils.atlas_plots import plot_calorimeter_shower, plot_calorimeter_shower_simplified


In [ ]:
hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(version_base=None, config_path="config")
cfg=compose(config_name="config.yaml")
wandb.init(tags = [cfg.data.dataset_name], project=cfg.wandb.project, entity=cfg.wandb.entity, config=OmegaConf.to_container(cfg, resolve=True), mode='disabled')
new_model = False
if new_model:
    self = setup_model(config)
else:
    config = OmegaConf.load(cfg.config_path)
    config.gpu_list = cfg.gpu_list
    config.load_state = cfg.load_state
    self = setup_model(config)
    self._model_creator.load_state(config.run_path, self.device)


In [ ]:
self.evaluate_ae(self.data_mgr.val_loader, 0)

In [ ]:
choice = 5000
print(self.incident_energy.shape)
plot_calorimeter_shower_simplified(
    gt_showers=self.showers, 
    showers_recon=self.showers_recon, 
    incident_energies=self.incident_energy, 
    choice=choice, 
    num_events=2, 
    cfg=cfg
)

In [ ]:
self.model.eval()
ar_input_size = self._config.data.z * self._config.data.r * self._config.data.phi
bs = [batch[0].shape[0] for batch in self.data_mgr.train_loader]
showers = torch.zeros(sum(bs), ar_input_size)
incident_energies = torch.zeros((sum(bs), 1))
with torch.no_grad():
    for i, (x, x0) in enumerate(self.data_mgr.train_loader):
        idx1, idx2 = int(np.sum(bs[:i])), int(np.sum(bs[:i+1]))
        incident_energies[idx1:idx2,:] = x0.cpu()
        showers[idx1:idx2,:] = x.cpu()



In [ ]:
visualize_low_ratio_events(showers=showers, 
                           e_inc=incident_energies,
                           binning_path=self._config.data.binning_path,
                           cutoff=0.5,
                           num_events=9
                           )

In [6]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# --- Data ---
models = ['GEANT4', 'CaloDream', 'CaloQVAE\n(GPU)', 'CaloQVAE\n(QPU)']
times_ms = [1000, 74.3, 2.0, 0.181]

# --- Style Setup ---
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.size'] = 14

colors = ["black",  '#648FFF', '#DC267F','#E66100', '#5D3A9B']

# 2. Define Patterns for Texture (Hatching)
#    / = diagonal, x = cross, * = stars, o = circles, etc.
patterns = ['/', 'x', '|', 'o']

fig, ax = plt.subplots(figsize=(10, 15)) # Increased height slightly for footnote

# --- Plotting ---
bars = ax.bar(models, times_ms, color=colors, width=0.7, zorder=3, edgecolor='white', linewidth=1.5)

# Apply patterns to bars individually
for bar, pattern in zip(bars, patterns):
    bar.set_hatch(pattern)

# --- Axis Formatting ---
ax.set_yscale('log')

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#888888')
ax.spines['bottom'].set_color('#333333')
ax.spines['bottom'].set_linewidth(1.5)

# Add horizontal gridlines
ax.grid(axis='y', which='major', linestyle='--', alpha=0.5, color='#cccccc', zorder=0)

# Custom Y-Axis Labels
ax.set_yticks([0.1, 1, 10, 100, 1000])
ax.get_yaxis().set_major_formatter(ticker.ScalarFormatter()) 
ax.set_yticklabels(['100 $\mu$s', '1 ms', '10 ms', '100 ms', '1 s'], fontsize=14, color='#555555')

# X-Axis Labels
ax.set_xticklabels(models, fontsize=20, color='#333333')

# 2. Iterate through the labels and target the last two
labels = ax.get_xticklabels()
for i, label in enumerate(labels):
    # If it's one of the last two labels
    if i >= len(models) - 2:
        label.set_fontweight('bold')
# Title and Label
ax.set_title('Generative Model Latency Comparison', fontsize=28, weight='bold', pad=30, loc='left')
ax.set_ylabel('Generation Time per Shower', fontsize=16, labelpad=10, color='#555555')

# --- Annotations ---
for i, (bar, value) in enumerate(zip(bars, times_ms)):
    height = bar.get_height()
    
    # 3. Logic for the Asterisk (Supervisor Request)
    is_calo_dream = (models[i] == 'CaloDream')
    
    # Text Formatting
    if value >= 1000:
        label = f"{value/1000:.0f} s"
    elif value < 1:
        label = f"{value*1000:.0f} $\mu$s"
    else:
        label = f"{value:.1f} ms"
        
    # Append asterisk if it is CaloDream
    if is_calo_dream:
        label += "*"

    # Make the "Best" result pop with bold color
    # For the orange bar (last one), use the orange color for text
    text_color = colors[i] if i == 3 else '#333333'
    font_weight = 'bold' if i == 3 else 'normal'
    
    ax.text(
        bar.get_x() + bar.get_width() / 2, 
        height * 1.2, 
        label,
        ha='center', 
        va='bottom', 
        fontsize=24, 
        fontweight=font_weight,
        color=text_color
    )

# --- Speedup Annotation ---
speedup = times_ms[0] / times_ms[-1]
ax.annotate(
    f'~{int(speedup):,}x Speedup',
    xy=(bars[-1].get_x() + bars[-1].get_width()/2, times_ms[-1]),
    xytext=(bars[-1].get_x() - 0.25, 1.5), 
    arrowprops=dict(arrowstyle='->', connectionstyle="arc3,rad=-0.2", color='#555555', lw=1.5),
    fontsize=18,
    color='#555555',
    fontstyle='italic'
)

# --- 4. Footnote (Supervisor Request) ---
footnote_text = (
    "* Timing from Calochallenge dataset 2 [2]"
)

# Place text relative to figure (0,0 is bottom left, 1,1 is top right)
# We adjust bottom margin to make room
plt.figtext(
    0.5, 0.05,  # x, y positions
    footnote_text, 
    ha="center", 
    fontsize=9, 
    color="#555555",
    style='italic'
)

# Adjust layout to ensure footnote isn't cut off
# plt.subplots_adjust(bottom=0.18)

# Save and Show
plt.savefig('model_latency_poster_version.svg', dpi=600, bbox_inches='tight')
plt.show()